# Elite Text Analytics Lab: Bangladesh Election Discourse

This notebook provides a high-end, interactive text analytics workflow:
- Multi-dataset ingestion and harmonization
- Quality diagnostics and sentiment reliability analysis
- Term intelligence (unigrams/bigrams, distinctive terms)
- Semantic landscape (interactive projection)
- Similar-comment mining
- Sarcasm behavior diagnostics
- Location intelligence with an interactive Bangladesh map
- Lightweight model benchmarking and explainability-friendly outputs

## 1. Setup

In [1]:
from pathlib import Path
import re
import json
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium.plugins import MarkerCluster
import ipywidgets as widgets
from IPython.display import display, HTML

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import networkx as nx

pd.set_option('display.max_colwidth', 220)

COLOR_MAP = {
    'Negative': '#d62728',
    'Sarcastic_negative': '#8b0000',
    'Neutral': '#7f7f7f',
    'Positive': '#2ca02c',
}

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'data'
OUT_DIR = ROOT / 'outputs' / 'notebook_assets'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Root:', ROOT)
print('Data dir:', DATA_DIR)

/Users/mdrafsunsheikh/Projects/bangladesh_election_pre_post_text_analysis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Root: /Users/mdrafsunsheikh/Projects/bangladesh_election_pre_post_text_analysis
Data dir: /Users/mdrafsunsheikh/Projects/bangladesh_election_pre_post_text_analysis/data


## 2. Load Final Annotated Datasets

In [2]:
def detect_text_col(df):
    for c in ['Comment', 'comment', 'text', 'Text']:
        if c in df.columns:
            return c
    obj_cols = [c for c in df.columns if df[c].dtype == 'object']
    return obj_cols[0] if obj_cols else df.columns[0]

# Accept both naming styles:
# - *.annotated.final.csv
# - * annotated.final.csv
# And fallback to completed files if a final file is missing.
final_files = sorted(
    set(DATA_DIR.glob('*.annotated.final.csv')) |
    set(DATA_DIR.glob('* annotated.final.csv'))
)

if not final_files:
    final_files = sorted(
        set(DATA_DIR.glob('*.annotated.completed.csv')) |
        set(DATA_DIR.glob('* annotated.completed.csv'))
    )

if not final_files:
    raise FileNotFoundError('No annotated final/completed CSV files found in data/.')

frames = []
for fp in final_files:
    df = pd.read_csv(fp)
    text_col = detect_text_col(df)
    dataset_name = fp.name.replace('.annotated.final.csv', '').replace(' annotated.final.csv', '')
    dataset_name = dataset_name.replace('.annotated.completed.csv', '').replace(' annotated.completed.csv', '')

    work = df.copy()
    work['dataset'] = dataset_name
    work['text_col_used'] = text_col
    work['comment'] = work[text_col].fillna('').astype(str)
    work['comment_len'] = work['comment'].str.len()
    work['token_len_est'] = work['comment'].str.split().str.len()

    if 'Sentiment' not in work.columns:
        raise ValueError(f'Sentiment column missing in {fp.name}')
    work['Sentiment'] = work['Sentiment'].astype(str).str.strip()

    if 'sentiment_confidence' not in work.columns:
        work['sentiment_confidence'] = np.nan
    if 'sentiment_source' not in work.columns:
        work['sentiment_source'] = 'unknown'
    if 'needs_review' not in work.columns:
        work['needs_review'] = False

    work['row_id'] = np.arange(len(work))
    work['file_name'] = fp.name
    frames.append(work)

all_df = pd.concat(frames, ignore_index=True)
all_df = all_df.replace([np.inf, -np.inf], np.nan)

print('Datasets loaded:', len(final_files))
print('Total rows:', len(all_df))
display(all_df[['dataset','file_name']].drop_duplicates().sort_values('dataset'))


Datasets loaded: 5
Total rows: 8224


,dataset,file_name
0,12.02.26 —- 17.02.26 - Sheet1,12.02.26 —- 17.02.26 - Sheet1.annotated.final.csv
747,18.02.26 —--- 01.03.26 - Sheet1,18.02.26 —--- 01.03.26 - Sheet1.annotated.final.csv
2402,After Election,After Election.annotated.final.csv
4337,After Forming Government,After Forming Government.annotated.final.csv
6854,Before Election some,Before Election some annotated.final.csv


## 3. Executive Snapshot (Interactive)

In [3]:
summary = (
    all_df.groupby('dataset', as_index=False)
    .agg(
        rows=('comment', 'size'),
        avg_len=('comment_len', 'mean'),
        median_len=('comment_len', 'median'),
        avg_tokens=('token_len_est', 'mean'),
        avg_confidence=('sentiment_confidence', 'mean'),
        sarcastic_share=('Sentiment', lambda s: (s=='Sarcastic_negative').mean()),
    )
    .sort_values('rows', ascending=False)
)

display(summary.style.background_gradient(cmap='Blues', subset=['rows','avg_confidence']))

fig = px.bar(
    summary,
    x='dataset', y='rows',
    color='avg_confidence',
    color_continuous_scale='Viridis',
    title='Dataset Volume with Mean Sentiment Confidence',
)
fig.update_layout(xaxis_tickangle=-20, height=500)
fig.show()

ImportError: background_gradient requires matplotlib.

## 4. Sentiment Intelligence Dashboard

In [4]:
sent_counts = (
    all_df.groupby(['dataset','Sentiment'], as_index=False)
    .size()
    .rename(columns={'size':'count'})
)
sent_counts['share'] = sent_counts['count'] / sent_counts.groupby('dataset')['count'].transform('sum')

fig = px.bar(
    sent_counts,
    x='dataset', y='share', color='Sentiment',
    title='Sentiment Mix by Dataset',
    color_discrete_map=COLOR_MAP,
)
fig.update_layout(height=520, xaxis_tickangle=-20, yaxis_tickformat='.0%')
fig.show()

fig2 = px.box(
    all_df.dropna(subset=['sentiment_confidence']),
    x='Sentiment', y='sentiment_confidence', color='Sentiment',
    color_discrete_map=COLOR_MAP,
    title='Confidence Distribution by Sentiment Class'
)
fig2.update_layout(height=500)
fig2.show()

## 6. Distinctive Terms by Dataset (TF-IDF Lens)

In [5]:
def top_terms_tfidf(frame, group_col='dataset', text_col='comment', min_df=3, top_k=20):
    rows = []
    groups = sorted(frame[group_col].dropna().unique())

    vec = TfidfVectorizer(ngram_range=(1,2), min_df=min_df, token_pattern=r'(?u)\b\w+\b')
    X = vec.fit_transform(frame[text_col].fillna(''))
    terms = np.array(vec.get_feature_names_out())

    for g in groups:
        idx = frame[group_col].values == g
        if idx.sum() == 0:
            continue
        mean_scores = np.asarray(X[idx].mean(axis=0)).ravel()
        top_idx = mean_scores.argsort()[::-1][:top_k]
        for rank, t_i in enumerate(top_idx, 1):
            rows.append({'group': g, 'rank': rank, 'term': terms[t_i], 'score': float(mean_scores[t_i])})
    return pd.DataFrame(rows)

terms_df = top_terms_tfidf(all_df, group_col='dataset', top_k=15)
display(terms_df.head(30))

fig = px.bar(
    terms_df,
    x='score', y='term',
    facet_col='group', facet_col_wrap=2,
    color='score', color_continuous_scale='Turbo',
    title='Top TF-IDF Terms per Dataset',
)
fig.update_layout(height=900)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.show()

,group,rank,term,score
0,12.02.26 —- 17.02.26 - Sheet1,1,র,0.080650
1,12.02.26 —- 17.02.26 - Sheet1,2,ক,0.066023
2,12.02.26 —- 17.02.26 - Sheet1,3,য,0.057555
3,12.02.26 —- 17.02.26 - Sheet1,4,ত,0.056436
4,12.02.26 —- 17.02.26 - Sheet1,5,দ,0.050457
5,12.02.26 —- 17.02.26 - Sheet1,6,ম,0.049463
6,12.02.26 —- 17.02.26 - Sheet1,7,ন,0.040985
7,12.02.26 —- 17.02.26 - Sheet1,8,ল,0.040636
8,12.02.26 —- 17.02.26 - Sheet1,9,য দ,0.039836
9,12.02.26 —- 17.02.26 - Sheet1,10,ধ,0.038909


## 7. Semantic Landscape (Interactive Projection)

In [6]:
sample_n = min(2200, len(all_df))
sem_df = all_df.sample(sample_n, random_state=42).copy()

vec = TfidfVectorizer(min_df=2, max_features=12000, ngram_range=(1,2), token_pattern=r'(?u)\b\w+\b')
X = vec.fit_transform(sem_df['comment'])

svd = TruncatedSVD(n_components=min(80, max(5, X.shape[1]-1)), random_state=42)
X_svd = svd.fit_transform(X)

perp = max(8, min(35, sample_n // 25))
tsne = TSNE(n_components=2, perplexity=perp, learning_rate='auto', init='random', random_state=42)
XY = tsne.fit_transform(X_svd)

sem_df['x'] = XY[:,0]
sem_df['y'] = XY[:,1]

fig = px.scatter(
    sem_df,
    x='x', y='y',
    color='Sentiment',
    symbol='dataset',
    color_discrete_map=COLOR_MAP,
    hover_data=['dataset','sentiment_confidence','comment'],
    title='Semantic Landscape of Comments (t-SNE on TF-IDF Embeddings)'
)
fig.update_traces(marker=dict(size=8, opacity=0.75))
fig.update_layout(height=760)
fig.show()

## 8. Similar-Comment Miner

In [7]:
sub = all_df[['dataset','comment','Sentiment']].copy()
sub = sub[sub['comment'].str.len() >= 8].drop_duplicates(subset=['comment']).reset_index(drop=True)
sub = sub.sample(min(1200, len(sub)), random_state=7)

vec = TfidfVectorizer(min_df=2, ngram_range=(1,2), token_pattern=r'(?u)\b\w+\b')
X = vec.fit_transform(sub['comment'])
S = cosine_similarity(X)

pairs = []
for i in range(S.shape[0]):
    j = np.argsort(S[i])[::-1]
    for k in j[1:4]:
        if i < k and S[i,k] >= 0.58:
            pairs.append((i,k,float(S[i,k])))

pairs_df = pd.DataFrame(pairs, columns=['i','j','similarity']).sort_values('similarity', ascending=False).head(40)
if len(pairs_df) == 0:
    display(HTML('<b>No high-similarity pairs found with current threshold.</b>'))
else:
    out = pairs_df.copy()
    out['comment_a'] = out['i'].map(sub['comment'])
    out['comment_b'] = out['j'].map(sub['comment'])
    out['dataset_a'] = out['i'].map(sub['dataset'])
    out['dataset_b'] = out['j'].map(sub['dataset'])
    display(out[['similarity','dataset_a','dataset_b','comment_a','comment_b']])

,similarity,dataset_a,dataset_b,comment_a,comment_b
7,1.000000,12.02.26 —- 17.02.26 - Sheet1,NaN,Please send them to Bangladesh with guard of honour. Kallu with recieve them with garlands....,NaN
131,1.000000,NaN,NaN,NaN,NaN
88,1.000000,NaN,NaN,NaN,NaN
89,1.000000,12.02.26 —- 17.02.26 - Sheet1,18.02.26 —--- 01.03.26 - Sheet1,শুভ কামনা রইল,আমার বঙ্গবন্ধু বলে তেলবাজী করা সারজিদ আলমের কষ্ট হওয়াটাই স্বাভাবিক
99,1.000000,NaN,NaN,NaN,NaN
123,1.000000,NaN,NaN,NaN,NaN
145,0.983537,NaN,NaN,NaN,NaN
140,0.948004,NaN,NaN,NaN,NaN
135,0.948004,NaN,NaN,NaN,NaN
124,0.948004,NaN,NaN,NaN,NaN


## 9. Sarcasm Behavior Diagnostics

In [8]:
def sarcasm_features(text):
    t = str(text)
    return {
        'exclaim': t.count('!'),
        'question': t.count('?'),
        'emoji_like': sum(ch in t for ch in ['😂','🤣','😏','🙃','🙂']),
        'quote_like': t.count('"') + t.count("'") + t.count('“') + t.count('”'),
        'len': len(t),
    }

feat = all_df['comment'].apply(sarcasm_features).apply(pd.Series)
sf = pd.concat([all_df[['Sentiment']], feat], axis=1)
agg = sf.groupby('Sentiment', as_index=False).mean(numeric_only=True)

display(agg)

radar_cols = ['exclaim','question','emoji_like','quote_like','len']
fig = go.Figure()
for senti in sorted(agg['Sentiment'].unique()):
    row = agg[agg['Sentiment']==senti].iloc[0]
    fig.add_trace(go.Scatterpolar(
        r=[row[c] for c in radar_cols],
        theta=radar_cols,
        fill='toself',
        name=senti,
        line=dict(color=COLOR_MAP.get(senti, '#1f77b4'))
    ))
fig.update_layout(title='Stylistic Signature by Sentiment (Sarcasm Lens)', height=620)
fig.show()

,Sentiment,exclaim,question,emoji_like,quote_like,len
0,Negative,0.052469,0.072531,0.0,0.110725,51.890432
1,Neutral,0.048815,0.093445,0.0,0.122734,45.355649
2,Positive,0.042900,0.053172,0.0,0.083384,51.986707
3,Sarcastic_negative,0.061963,0.083129,0.0,0.066871,30.877914


## 10. Interactive Bangladesh Location Map

This map is built from rows that contain location columns (`Location 1/2/3`).
Bubble size = mentions, color intensity = sarcastic+negative pressure.

In [9]:
DISTRICT_COORDS = {
    'Dhaka': (23.8103, 90.4125), 'Gazipur': (24.0023, 90.4264), 'Narayanganj': (23.6238, 90.5000),
    'Narsingdi': (23.9220, 90.7177), 'Tangail': (24.2513, 89.9167), 'Manikganj': (23.8617, 90.0003),
    'Munshiganj': (23.5422, 90.5305), 'Rajbari': (23.7574, 89.6445), 'Faridpur': (23.6070, 89.8429),
    'Gopalganj': (23.0051, 89.8266), 'Madaripur': (23.1641, 90.1897), 'Shariatpur': (23.2423, 90.4348),
    'Kishoreganj': (24.4449, 90.7766), 'Mymensingh': (24.7471, 90.4203), 'Jamalpur': (24.9375, 89.9378),
    'Sherpur': (25.0205, 90.0153), 'Netrokona': (24.8835, 90.7279), 'Chattogram': (22.3569, 91.7832),
    "Cox's Bazar": (21.4272, 92.0058), 'Rangamati': (22.7324, 92.2985), 'Bandarban': (22.1953, 92.2184),
    'Khagrachhari': (23.1193, 91.9847), 'Cumilla': (23.4607, 91.1809), 'Brahmanbaria': (23.9571, 91.1117),
    'Chandpur': (23.2333, 90.6713), 'Feni': (23.0159, 91.3976), 'Noakhali': (22.8246, 91.1017),
    'Lakshmipur': (22.9447, 90.8282), 'Sylhet': (24.8949, 91.8687), 'Moulvibazar': (24.4829, 91.7774),
    'Habiganj': (24.3745, 91.4155), 'Sunamganj': (25.0658, 91.3950), 'Rajshahi': (24.3745, 88.6042),
    'Naogaon': (24.7936, 88.9318), 'Natore': (24.4206, 89.0003), 'Chapainawabganj': (24.5965, 88.2775),
    'Pabna': (24.0064, 89.2372), 'Sirajganj': (24.4534, 89.7007), 'Bogura': (24.8465, 89.3776),
    'Joypurhat': (25.0968, 89.0230), 'Rangpur': (25.7439, 89.2752), 'Gaibandha': (25.3290, 89.5403),
    'Kurigram': (25.8054, 89.6362), 'Lalmonirhat': (25.9923, 89.2847), 'Nilphamari': (25.9318, 88.8560),
    'Dinajpur': (25.6279, 88.6332), 'Thakurgaon': (26.0337, 88.4617), 'Panchagarh': (26.3411, 88.5542),
    'Khulna': (22.8456, 89.5403), 'Jessore': (23.1664, 89.2081), 'Jhenaidah': (23.5448, 89.1539),
    'Magura': (23.4855, 89.4198), 'Narail': (23.1725, 89.5127), 'Satkhira': (22.7185, 89.0705),
    'Bagerhat': (22.6516, 89.7859), 'Kushtia': (23.9013, 89.1205), 'Chuadanga': (23.6402, 88.8418),
    'Meherpur': (23.7622, 88.6318), 'Barishal': (22.7010, 90.3535), 'Bhola': (22.6859, 90.6482),
    'Patuakhali': (22.3596, 90.3296), 'Pirojpur': (22.5781, 89.9787), 'Jhalokati': (22.6406, 90.1987),
    'Barguna': (22.1592, 90.1250),
}

ALIASES = {
    'barisal':'Barishal','বরিশাল':'Barishal', 'chittagong':'Chattogram','চট্টগ্রাম':'Chattogram',
    'comilla':'Cumilla','কুমিল্লা':'Cumilla','bogra':'Bogura','বগুড়া':'Bogura','বগুড়া':'Bogura',
    'jashore':'Jessore','যশোর':'Jessore','নড়াইল':'Narail','নড়াইল':'Narail',
}


def norm_loc(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s:
        return None
    k = re.sub(r'\s+', ' ', s.lower()).replace(' জেলা','').strip()
    if k in ALIASES:
        return ALIASES[k]
    if s in DISTRICT_COORDS:
        return s
    t = s.title()
    if t in DISTRICT_COORDS:
        return t
    return ALIASES.get(s, s)

loc_cols = [c for c in ['Location 1','Location 2','Location 3'] if c in all_df.columns]
if not loc_cols:
    display(HTML('<b>No location columns found in current loaded final files.</b>'))
else:
    parts = []
    for c in loc_cols:
        part = all_df[['dataset','comment','Sentiment',c]].rename(columns={c:'location_raw'})
        parts.append(part)
    loc_df = pd.concat(parts, ignore_index=True)
    loc_df['location'] = loc_df['location_raw'].map(norm_loc)
    loc_df = loc_df[loc_df['location'].notna()].copy()

    pressure = (
        loc_df.assign(is_pressure=loc_df['Sentiment'].isin(['Negative','Sarcastic_negative']).astype(int))
        .groupby('location', as_index=False)
        .agg(
            mentions=('location','size'),
            pressure_share=('is_pressure','mean'),
            sample_comment=('comment','first')
        )
    )
    pressure['lat'] = pressure['location'].map(lambda z: DISTRICT_COORDS.get(z, (None,None))[0])
    pressure['lon'] = pressure['location'].map(lambda z: DISTRICT_COORDS.get(z, (None,None))[1])
    pressure = pressure[pressure['lat'].notna()].sort_values('mentions', ascending=False)

    m = folium.Map(location=[23.7, 90.4], zoom_start=7, tiles='cartodbpositron')
    cl = MarkerCluster().add_to(m)

    max_m = max(pressure['mentions'].max(), 1)
    for r in pressure.itertuples(index=False):
        radius = 5 + 22 * ((r.mentions / max_m) ** 0.5)
        red = int(255 * r.pressure_share)
        green = int(180 * (1-r.pressure_share))
        color = f'#{red:02x}{green:02x}55'
        popup = f"<b>{r.location}</b><br>Mentions: {r.mentions}<br>Neg+Sarcastic Share: {r.pressure_share:.1%}<br><i>{str(r.sample_comment)[:180]}</i>"
        folium.CircleMarker(
            location=[r.lat, r.lon], radius=radius,
            color=color, fill=True, fill_opacity=0.55,
            popup=folium.Popup(popup, max_width=350)
        ).add_to(cl)

    map_path = OUT_DIR / 'bangladesh_interactive_location_map.html'
    m.save(str(map_path))
    display(m)
    print('Saved interactive map to:', map_path)

    display(pressure.head(25))

Saved interactive map to: /Users/mdrafsunsheikh/Projects/bangladesh_election_pre_post_text_analysis/outputs/notebook_assets/bangladesh_interactive_location_map.html


,location,mentions,pressure_share,sample_comment,lat,lon
13,Dhaka,1508,0.771220,"""ওখানে ধরা খাবে তারপর সরকারিভাবে দেশের কারাগারে আসবে, দেশের আদালত মুক্ত করে দিবে। এটাই পলিটিক্স।""",23.8103,90.4125
9,Chattogram,363,0.798898,নিজ দেশেও (ভারত) নিরাপদ নয় আওয়ামীলীগ,22.3569,91.7832
3,Barishal,166,0.825301,খেলা শুরু এভাবেই চলবে,22.7010,90.3535
26,Khulna,141,0.773050,"""নাটক কইরে দেশে পাঠাবে, পাকড়াও দেখায় খালাশ। মাঝখানে কিছু টাকার ছড়াছড়ি!""",22.8456,89.5403
12,Cumilla,102,0.774510,আল্লাহরে এডা কি হইলো,23.4607,91.1809
60,Sylhet,97,0.835052,এগুলোকে এখন গ্রেফতার করবে আর আস্তে আস্তে সবগুলোকে বাংলাদেশ প্রেরণ করবে,24.8949,91.8687
18,Gazipur,93,0.849462,"""ভাগ্য ভালো দলের সিদ্ধান্তে হয়েছেন, নয়তো নিজে নিজেই মন্ত্রিত্বের শপথ নিতেন.!""",24.0023,90.4264
38,Mymensingh,81,0.790123,ওরে বাটপার,24.7471,90.4203
54,Rangpur,70,0.728571,খেলা হবে নিরাপদে,25.7439,89.2752
46,Noakhali,69,0.768116,আমার পেইজটিতে আমন্ত্রণ রইলো,22.8246,91.1017


## 11. Location Co-occurrence Network (Interactive)

In [10]:
if loc_cols:
    row_locs = (
        all_df[['dataset','comment'] + loc_cols]
        .copy()
    )

    edges = Counter()
    for _, row in row_locs.iterrows():
        locs = []
        for c in loc_cols:
            z = norm_loc(row[c])
            if z and z in DISTRICT_COORDS:
                locs.append(z)
        locs = sorted(set(locs))
        for a, b in combinations(locs, 2):
            edges[(a,b)] += 1

    edge_df = pd.DataFrame([(a,b,w) for (a,b),w in edges.items()], columns=['a','b','weight'])
    edge_df = edge_df.sort_values('weight', ascending=False).head(120)

    if len(edge_df) == 0:
        display(HTML('<b>No co-occurrence edges found.</b>'))
    else:
        G = nx.Graph()
        for r in edge_df.itertuples(index=False):
            G.add_edge(r.a, r.b, weight=r.weight)

        pos = nx.spring_layout(G, seed=42, k=0.65)

        edge_x, edge_y = [], []
        for u, v, d in G.edges(data=True):
            x0, y0 = pos[u]
            x1, y1 = pos[v]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]

        node_x, node_y, node_text, node_size = [], [], [], []
        for n in G.nodes():
            x, y = pos[n]
            deg = G.degree(n)
            wt = sum(G[n][nbr]['weight'] for nbr in G.neighbors(n))
            node_x.append(x)
            node_y.append(y)
            node_text.append(f"{n}<br>Degree: {deg}<br>Total Edge Weight: {wt}")
            node_size.append(10 + 2.2 * deg)

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(width=0.8, color='#9ca3af'), hoverinfo='none'))
        fig.add_trace(go.Scatter(
            x=node_x, y=node_y, mode='markers+text',
            marker=dict(size=node_size, color='#1d4ed8', opacity=0.85),
            text=[t.split('<br>')[0] for t in node_text], textposition='top center',
            hovertext=node_text, hoverinfo='text'
        ))
        fig.update_layout(title='District Co-occurrence Network', height=760, showlegend=False)
        fig.show()

        display(edge_df.head(25))

,a,b,weight
0,Dhaka,Gazipur,84
24,Chattogram,Dhaka,81
8,Chattogram,Noakhali,59
17,Barishal,Dhaka,59
9,Dhaka,Mymensingh,54
38,Dhaka,Khulna,46
29,Dhaka,Rangpur,39
37,Dhaka,Kishoreganj,36
3,Dhaka,Pabna,32
67,Jessore,Khulna,31


## 12. Model Bench (Fast Baseline)
This gives a practical estimate of class separability in your current labeled corpus.

In [11]:
bench = all_df[['comment','Sentiment']].dropna().copy()
bench = bench[bench['comment'].str.len() > 0]

X_train, X_test, y_train, y_test = train_test_split(
    bench['comment'], bench['Sentiment'],
    test_size=0.2, random_state=42, stratify=bench['Sentiment']
)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2, ngram_range=(1,2), token_pattern=r'(?u)\b\w+\b')),
    ('clf', LogisticRegression(max_iter=1400, class_weight='balanced'))
])
pipe.fit(X_train, y_train)

pred = pipe.predict(X_test)
report = classification_report(y_test, pred, output_dict=True, zero_division=0)
rep_df = pd.DataFrame(report).T

display(rep_df)

labels = ['Negative','Sarcastic_negative','Neutral','Positive']
cm = confusion_matrix(y_test, pred, labels=labels)
cm_fig = px.imshow(cm, x=labels, y=labels, text_auto=True, color_continuous_scale='Blues',
                   title='Confusion Matrix (Baseline Text Classifier)')
cm_fig.update_layout(xaxis_title='Predicted', yaxis_title='True', height=520)
cm_fig.show()

,precision,recall,f1-score,support
Negative,0.797688,0.797688,0.797688,519.000000
Neutral,0.771429,0.765957,0.768683,141.000000
Positive,0.783133,0.785498,0.784314,331.000000
Sarcastic_negative,0.812883,0.812883,0.812883,652.000000
accuracy,0.798539,0.798539,0.798539,0.798539
macro avg,0.791283,0.790507,0.790892,1643.000000
weighted avg,0.798532,0.798539,0.798535,1643.000000


## 13. Save Curated Outputs

In [13]:
# Save high-value tables for downstream use.
summary.to_csv(OUT_DIR / 'dataset_summary.csv', index=False)
sent_counts.to_csv(OUT_DIR / 'sentiment_mix_by_dataset.csv', index=False)
terms_df.to_csv(OUT_DIR / 'top_terms_by_dataset_tfidf.csv', index=False)

print('Saved assets under:', OUT_DIR)
print(''.join(sorted([p.name for p in OUT_DIR.glob('*')])))

Saved assets under: /Users/mdrafsunsheikh/Projects/bangladesh_election_pre_post_text_analysis/outputs/notebook_assets
bangladesh_interactive_location_map.htmldataset_summary.csvsentiment_mix_by_dataset.csvtop_terms_by_dataset_tfidf.csv


## 14. What To Do Next
- Re-run this notebook after each new annotation wave.
- Promote low-confidence triage rows into a dedicated active-learning loop.
- Add temporal slicing once precise timestamps are available.
- Connect a retrieval-augmented query panel over comments for analyst Q&A.